# Phase 2 — Fit 3DGS to the with-object renders
Trains `splatfacto` on 70 orbit views, evaluates on 11 held-out views.
Runtime: GPU. After the install cell, restart the runtime and re-run cells 1–3.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy dataset from Drive to local disk (guard checks completeness, not existence)
import os, shutil
DRIVE = '/content/drive/MyDrive/light-footprint-removal'
DATA = '/content/data/footprint'
if not os.path.exists(f'{DATA}/transforms.json'):
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(f'{DRIVE}/renders/footprint_dataset', DATA)
print('frames:', len(os.listdir(f'{DATA}/with/rgb')))

In [ ]:
# Train/val/test splits: every 8th frame held out for novel-view evaluation
import json
root = f'{DATA}/with'
meta = json.load(open(f'{DATA}/transforms.json'))
frames = meta['frames']
test_idx = set(range(0, len(frames), 8))
splits = {'train': [f for i, f in enumerate(frames) if i not in test_idx],
          'test':  [f for i, f in enumerate(frames) if i in test_idx]}
splits['val'] = splits['test']
for name, fr in splits.items():
    json.dump({'camera_angle_x': meta['camera_angle_x'], 'frames': fr},
              open(f'{root}/transforms_{name}.json', 'w'))
print({k: len(v) for k, v in splits.items()})

In [ ]:
# Install (once). If Colab asks to restart: restart, re-run cells 1-3, skip this.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
!pip -q install nerfstudio

In [ ]:
# Train 3DGS (~15 min on A100)
!ns-train splatfacto --data /content/data/footprint/with \
  --output-dir /content/outputs --max-num-iterations 15000 \
  --viewer.quit-on-train-completion True --vis tensorboard blender-data

In [ ]:
import glob, os
CONFIG = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True),
                key=os.path.getmtime)[-1]
print(CONFIG)

In [ ]:
# Held-out metrics (env var: trust our own checkpoint under PyTorch>=2.6)
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-eval --load-config "$CONFIG" \
  --output-path /content/eval_with.json
import json
print(json.load(open('/content/eval_with.json'))['results'])

In [ ]:
# Render held-out views; compare one against ground truth
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset --load-config "$CONFIG" \
  --split test --output-path /content/renders_test
import glob
from PIL import Image
import matplotlib.pyplot as plt
rgb = sorted(glob.glob('/content/renders_test/test/rgb/*.jpg'))
gt  = sorted(glob.glob('/content/renders_test/test/gt-rgb/*.jpg'))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open(rgb[5])); ax[0].set_title('3DGS render (held-out view)')
ax[1].imshow(Image.open(gt[5]));  ax[1].set_title('ground truth')
for a in ax: a.axis('off')
plt.show()

In [ ]:
# Persist model + metrics to Drive
import shutil, os
OUT = f'{DRIVE}/checkpoints/phase2_with'
os.makedirs(OUT, exist_ok=True)
shutil.copytree('/content/outputs', f'{OUT}/outputs', dirs_exist_ok=True)
shutil.copy('/content/eval_with.json', f'{OUT}/eval_with.json')
print('saved to', OUT)